# Model Architecture: EfficientNetV2-S

Notebook ini berfokus pada pendefinisian arsitektur model, fungsi pembeku (freeze) untuk fine-tuning 2 tahap, fungsi loss dengan pembobotan kelas, optimizer, scheduler, dan logika Early Stopping berbasis Macro F1.

**Spesifikasi Model:**
- Arsitektur Dasar: `tf_efficientnetv2_s` dari `timm` (Pretrained pada ImageNet)
- Resolusi Input: 256x256
- Jumlah Kelas: 3 (Recyclable, Electronic, Organic)

In [ ]:
# 1. Setup & Imports
import torch
import torch.nn as nn
import torch.optim as optim
import timm
import numpy as np

print(f"PyTorch Version: {torch.__version__}")
print(f"Timm Version: {timm.__version__}")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Menggunakan device: {DEVICE}")

In [ ]:
# 2. Arsitektur Model: EfficientNetV2-S & Two-Stage Fine-Tuning Setup

class SampahClassifier(nn.Module):
    def __init__(self, num_classes=3, pretrained=True):
        super(SampahClassifier, self).__init__()
        # Inisialisasi EfficientNetV2-S dari timm
        # Menggunakan 'tf_efficientnetv2_s' yang terbukti sangat baik di image size ~300-400, namun kita pakai 256
        self.backbone = timm.create_model('tf_efficientnetv2_s', pretrained=pretrained, num_classes=0) # num_classes=0 membuang head lama
        
        # Mengambil jumlah fitur output dari backbone
        num_features = self.backbone.num_features
        
        # Custom Head / Classifier untuk 3 kelas
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.2), # Dropout standar untuk regularisasi
            nn.Linear(num_features, num_classes)
        )

    def forward(self, x):
        features = self.backbone(x)
        out = self.classifier(features)
        return out

    def freeze_backbone(self):
        """
        Stage 1 Fine-Tuning: 
        Bekukan seluruh parameter backbone, hanya latih bagian classifier / head baru.
        Tujuannya agar random initialization head tidak merusak fitur pretrained.
        """
        for param in self.backbone.parameters():
            param.requires_grad = False
        for param in self.classifier.parameters():
            param.requires_grad = True
        print("Backbone FROZEN. Hanya melatih classifier head.")

    def unfreeze_backbone(self):
        """
        Stage 2 Fine-Tuning:
        Buka kunci seluruh parameter backbone untuk fine-tuning penuh (biasanya dengan LR lebih kecil).
        """
        for param in self.parameters():
            param.requires_grad = True
        print("Backbone UNFROZEN. Melatih seluruh model.")

# Inisialisasi Model untuk testing (bisa dihapus/dikomen nantinya)
model = SampahClassifier(num_classes=3)
model.to(DEVICE)
print(f"Model berhasil dibuat! Input resolution target: 256x256")

In [ ]:
# 3. Fungsi Loss dengan Class Weights

def create_loss_function(class_counts):
    """
    Membuat Weighted CrossEntropyLoss untuk menangani Class Imbalance.
    class_counts: List atau array jumlah gambar per kelas, misal: [9990, 7967, 12545]
    """
    total_samples = sum(class_counts)
    num_classes = len(class_counts)
    
    # Formula standard untuk pembobotan kelas seimbang: 
    # weight = Total / (Num_Classes * Count)
    weights = [total_samples / (num_classes * count) for count in class_counts]
    weights_tensor = torch.tensor(weights, dtype=torch.float32).to(DEVICE)
    
    print(f"Class Weights yang digunakan: {weights_tensor.cpu().numpy()}")
    
    criterion = nn.CrossEntropyLoss(weight=weights_tensor)
    return criterion

# Contoh penggunaan berdasarkan manifest (Recyclable, Electronic, Organic)
# Angka ini berasal dari train_manifest.csv yang kita generate di Preprocessing.
class_counts_example = [9990, 7967, 12545]  
criterion = create_loss_function(class_counts_example)

In [ ]:
# 4. Optimizer AdamW & Scheduler Cosine Annealing (Setup Dummy)

def setup_optimizer_scheduler(model, stage=1, base_lr=1e-3, epochs=5):
    """
    Membuat optimizer AdamW dan CosineAnnealingLR scheduler.
    Stage 1 biasanya menggunakan LR lebih besar karena hanya melatih head.
    Stage 2 (unfrozen) harus menggunakan LR jauh lebih kecil (misal 1e-4) agar tidak merusak pretrained weights.
    """
    # Filter parameter yang membutuhkan gradient saja (penting saat Stage 1/frozen)
    trainable_params = filter(lambda p: p.requires_grad, model.parameters())
    
    optimizer = optim.AdamW(trainable_params, lr=base_lr, weight_decay=1e-2)
    
    # T_max adalah maksimum iterasi/epoch, eta_min adalah batas bawah LR
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    
    return optimizer, scheduler

# Contoh penggunaan untuk STAGE 1
model.freeze_backbone()
optimizer_s1, scheduler_s1 = setup_optimizer_scheduler(model, stage=1, base_lr=1e-3, epochs=5)
print(f"Stage 1 Optimizer & Scheduler siap.")

# Contoh penggunaan untuk STAGE 2 (simulasi)
# model.unfreeze_backbone()
# optimizer_s2, scheduler_s2 = setup_optimizer_scheduler(model, stage=2, base_lr=1e-4, epochs=20)

In [ ]:
# 5. Early Stopping berbasis Macro F1-Score

class EarlyStopping:
    def __init__(self, patience=7, mode='max', delta=0.0):
        """
        Early Stopping berdasarkan metrik validasi (Macro F1 Score).
        patience: Berapa epoch ditunggu sebelum stop jika tidak ada peningkatan.
        mode: 'max' untuk metrik yang ingin dimaksimalkan (seperti F1).
        delta: Minimum perubahan agar dianggap sebagai peningkatan.
        """
        self.patience = patience
        self.mode = mode
        self.delta = delta
        self.best_score = None
        self.counter = 0
        self.early_stop = False
        
    def __call__(self, current_score, model_state, save_path='best_model.pth'):
        if self.best_score is None:
            self.best_score = current_score
            self.save_checkpoint(current_score, model_state, save_path)
        else:
            if self.mode == 'max':
                is_better = current_score > (self.best_score + self.delta)
            else:
                is_better = current_score < (self.best_score - self.delta)
                
            if is_better:
                self.best_score = current_score
                self.counter = 0
                self.save_checkpoint(current_score, model_state, save_path)
            else:
                self.counter += 1
                print(f"EarlyStopping counter: {self.counter} out of {self.patience}")
                if self.counter >= self.patience:
                    self.early_stop = True
                    print("Early Stopping diaktifkan!")

    def save_checkpoint(self, score, model_state, save_path):
        print(f"Menemukan model terbaik dengan skor baru: {score:.4f}. Menyimpan model...")
        torch.save(model_state, save_path)

# Contoh instansiasi
# early_stopping = EarlyStopping(patience=5, mode='max')